## Cell 1: Setup
**What this demonstrates**: Environment initialisation for RAPTOR — LangChain for chunking and vector storage, scikit-learn for KMeans clustering, OpenAI for embeddings, Anthropic for cluster summarisation and answer generation.
**Expected output**: Setup confirmation with model names, chunk settings, and tree construction parameters.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb anthropic openai python-dotenv scikit-learn numpy llama-index

import os
import time
import pathlib
from collections import defaultdict

import numpy as np
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

ANSWER_MODEL  = 'claude-sonnet-4-6'
SUMMARY_MODEL = 'claude-haiku-4-5-20251001'  # cheap model for cluster summaries
EMBED_MODEL   = 'text-embedding-3-small'
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60
MIN_NODES     = 3    # stop recursing when fewer nodes remain than this
MAX_K         = 6    # maximum clusters per tree level
K_RETRIEVE    = 6    # nodes retrieved from the unified index at query time

client        = Anthropic()
lc_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Answer model  : {ANSWER_MODEL}')
print(f'  Summary model : {SUMMARY_MODEL}')
print(f'  Embed model   : {EMBED_MODEL}')
print(f'  Chunk size    : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap')
print(f'  Min nodes     : {MIN_NODES} (recursion stops below this)')
print(f'  Max K         : {MAX_K} clusters per level')

## Cell 2: Data — Earnings Report
**What this demonstrates**: Loading an earnings report as a mini hierarchical document. The report has natural section-level structure (Retail Banking, Commercial Banking, Investment Banking, Consolidated Metrics, Guidance) that RAPTOR can discover through clustering — without being told the sections exist.
**Expected output**: Character count, approximate section breakdown, and explanation of why hierarchical retrieval matters for this document type.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

earnings_text = (BASE_DIR / 'earnings_report.txt').read_text(encoding='utf-8')

# Detect natural sections — the document uses '=' * 80 as section dividers
sections = [s.strip() for s in earnings_text.split('=' * 80) if s.strip()]

print(f'Loaded earnings_report.txt: {len(earnings_text):,} characters')
print(f'Natural sections: {len(sections)}')
print()
print('Why RAPTOR suits this document:')
print('  Query A (specific) : "What was net interest income in Q3?"')
print('    -> needs leaf chunk with the $2,341M figure')
print('  Query B (thematic) : "What is the overall business outlook?"')
print('    -> needs a synthesis of guidance, segment growth, CEO commentary')
print('    -> no single leaf chunk contains this; flat retrieval fails')
print('  RAPTOR builds cross-section summary nodes for Query B')
print('  while keeping leaf chunks for Query A — one index serves both.')
print()
print('Section preview:')
for i, s in enumerate(sections, 1):
    first_line = s.split('\n')[0].strip()
    print(f'  {i}. {first_line[:70]}')

## Cell 3: Core — RAPTOR Tree Construction and Unified Index
**What this demonstrates**: Five functions implement the full RAPTOR pipeline: chunk documents into leaf nodes, select optimal K via silhouette score, summarise clusters with the LLM, recursively build the tree until a root is reached, and index all nodes (leaf + branch + root) into a single Chroma collection.
**Expected output**: Tree construction log showing each level's node count and cluster count, followed by the unified index summary.

In [ ]:
def chunk_document(text: str) -> list[str]:
    """Split a document into leaf chunks — the base level of the RAPTOR tree.

    Args:
        text: Raw document text.

    Returns:
        List of text chunks at tree level 0.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=['\n\n', '\n', '.', ' '],
    )
    return [d.page_content for d in splitter.create_documents([text])]


def optimal_k(emb_matrix: np.ndarray) -> int:
    """Select K for KMeans using silhouette score.

    Silhouette score (-1 to 1) measures how well each point fits its
    assigned cluster vs. the nearest alternative. Higher is better.
    Trying K from 2 to min(MAX_K, n-1) and picking the best score.

    Args:
        emb_matrix: (n_samples, n_features) embedding array.

    Returns:
        Optimal number of clusters K (1 if clustering is not meaningful).
    """
    n = len(emb_matrix)
    if n < 2:
        return 1
    best_k, best_score = 2, -1.0
    for k in range(2, min(MAX_K + 1, n)):
        labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb_matrix)
        if len(set(labels)) < 2:
            continue  # degenerate: all points in one cluster
        score = silhouette_score(emb_matrix, labels)
        if score > best_score:
            best_score, best_k = score, k
    return best_k


def summarize_cluster(texts: list[str], level: int) -> str:
    """Summarise a cluster of texts into a single RAPTOR tree node.

    Uses SUMMARY_MODEL (Haiku) — one call per cluster per level.
    The instruction varies by level: leaf clusters get a thematic overview;
    higher-level clusters get a synthesis across summaries.

    Args:
        texts: Texts (chunks or lower-level summaries) in this cluster.
        level: Tree level being summarised (0 = summarising leaf chunks).

    Returns:
        2-3 sentence summary string.
    """
    combined = '\n\n'.join(texts)
    instruction = (
        'Summarise these document excerpts into a coherent thematic overview.'
        if level == 0
        else 'Synthesise these section summaries into a higher-level thematic synthesis.'
    )
    response = client.messages.create(
        model=SUMMARY_MODEL,
        max_tokens=300,
        messages=[{
            'role': 'user',
            'content': (
                f'{instruction}\n\n'
                f'Text:\n{combined}\n\n'
                'Provide a 2-3 sentence summary capturing the main themes.'
            ),
        }],
    )
    return response.content[0].text.strip()


def build_raptor_tree(
    texts: list[str],
    level: int = 0,
) -> list[dict]:
    """Recursively build a RAPTOR tree from a list of texts.

    At each level:
      1. Record all input texts as nodes at this level.
      2. Embed, cluster (KMeans, silhouette-optimal K), summarise each cluster.
      3. Recurse on the summaries as the next level's input.

    Recursion stops when fewer than MIN_NODES texts remain — those are
    collapsed into a single root summary node.

    Args:
        texts: Text content at the current tree level.
        level: Current depth (0 = leaf chunks from the original document).

    Returns:
        Flat list of ALL tree nodes across all levels.
        Each node: {text, level, node_id}.
    """
    nodes: list[dict] = []

    # Record every input text as a node at this level
    for i, t in enumerate(texts):
        nodes.append({'text': t, 'level': level, 'node_id': f'L{level}N{i}'})

    # Stopping condition: too few texts to form meaningful clusters
    if len(texts) < MIN_NODES:
        if len(texts) >= 2:
            # Collapse remaining nodes into a single root summary
            root_text = summarize_cluster(texts, level=level)
            root_id = f'L{level + 1}N0'
            nodes.append({'text': root_text, 'level': level + 1, 'node_id': root_id})
            print(f'  Level {level + 1}: root node from {len(texts)} inputs ({root_id})')
        return nodes

    # Embed all texts at this level for clustering
    print(f'  Level {level}: embedding {len(texts)} texts...', end=' ', flush=True)
    emb_matrix = np.array(lc_embeddings.embed_documents(texts))
    k = optimal_k(emb_matrix)
    print(f'K={k}')

    if k <= 1:
        # Silhouette prefers a single cluster — collapse to root
        root_text = summarize_cluster(texts, level=level)
        nodes.append({'text': root_text, 'level': level + 1, 'node_id': f'L{level + 1}N0'})
        print(f'  Level {level + 1}: root node (single-cluster collapse)')
        return nodes

    # Assign each text to its nearest centroid
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb_matrix)

    cluster_groups: dict[int, list[str]] = defaultdict(list)
    for text, label in zip(texts, labels):
        cluster_groups[int(label)].append(text)

    # Summarise each cluster — each summary becomes a node at level+1
    print(f'  Level {level}: summarising {k} clusters...', end=' ', flush=True)
    next_level_texts: list[str] = []
    for cluster_id in sorted(cluster_groups):
        summary = summarize_cluster(cluster_groups[cluster_id], level=level)
        next_level_texts.append(summary)
    print('done')

    # Recurse: the summaries are the input texts for level+1
    upper_nodes = build_raptor_tree(next_level_texts, level=level + 1)
    nodes.extend(upper_nodes)
    return nodes


def build_unified_index(nodes: list[dict], collection_name: str = 'raptor_tree') -> Chroma:
    """Embed all RAPTOR nodes and insert into a single Chroma collection.

    Leaf chunks, intermediate summaries, and the root node all land in the
    same vector store. At query time the retriever matches any level.

    Args:
        nodes:           Flat node list from build_raptor_tree.
        collection_name: Chroma collection name — use distinct names per tree.

    Returns:
        Chroma vectorstore containing all nodes tagged with level metadata.
    """
    docs = [
        Document(
            page_content=node['text'],
            metadata={'level': node['level'], 'node_id': node['node_id']},
        )
        for node in nodes
    ]
    return Chroma.from_documents(docs, lc_embeddings, collection_name=collection_name)


def raptor_rag(query: str, vectorstore: Chroma, k: int = K_RETRIEVE) -> dict:
    """Retrieve from the unified RAPTOR index and generate an answer.

    Uses collapsed-tree retrieval: all nodes live in one vector store.
    Thematic queries naturally surface summary/root nodes; specific queries
    surface leaf nodes. Both can appear in the same result set.

    Args:
        query:       User question.
        vectorstore: Unified Chroma index from build_unified_index.
        k:           Number of nodes to retrieve.

    Returns:
        dict with keys: query, retrieved, level_breakdown, context, answer, latency_ms.
    """
    t0 = time.perf_counter()

    results = vectorstore.similarity_search_with_score(query, k=k)

    # Group retrieved nodes by tree level
    level_groups: dict[int, list] = defaultdict(list)
    for doc, score in results:
        level_groups[doc.metadata['level']].append((doc, score))

    max_level = max(level_groups.keys()) if level_groups else 0

    # Build generation context — higher tree levels first so the LLM
    # sees thematic synthesis before leaf-level details
    context_parts: list[str] = []
    for lvl in sorted(level_groups.keys(), reverse=True):
        if lvl == max_level:
            label = 'Cross-section synthesis (root)'
        elif lvl > 0:
            label = f'Level-{lvl} cluster summary'
        else:
            label = 'Source detail (leaf chunk)'
        for doc, score in level_groups[lvl]:
            context_parts.append(f'[{label} | score={score:.4f}]\n{doc.page_content}')

    context = '\n\n---\n\n'.join(context_parts)

    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=500,
        system=(
            'Answer using only the provided context. '
            'Context includes high-level thematic summaries and specific source details. '
            'Synthesise across levels for a complete answer.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )

    return {
        'query'           : query,
        'retrieved'       : results,
        'level_breakdown' : {lvl: len(docs) for lvl, docs in level_groups.items()},
        'context'         : context,
        'answer'          : response.content[0].text.strip(),
        'latency_ms'      : (time.perf_counter() - t0) * 1000,
    }


# ── Build the tree and unified index for the earnings report ──────────────────

print('Building RAPTOR tree for earnings_report.txt...')
leaf_chunks = chunk_document(earnings_text)
print(f'Leaf chunks (level 0): {len(leaf_chunks)}')
print()

all_nodes = build_raptor_tree(leaf_chunks)
vectorstore = build_unified_index(all_nodes, collection_name='raptor_earnings')

level_counts: dict[int, int] = defaultdict(int)
for node in all_nodes:
    level_counts[node['level']] += 1

max_lvl = max(level_counts.keys())

print()
print('Tree complete. Unified index ready.')
print(f'  Total nodes in index: {len(all_nodes)}')
for lvl in sorted(level_counts.keys()):
    label = 'leaf' if lvl == 0 else ('root' if lvl == max_lvl else 'summaries')
    print(f'  Level {lvl} ({label}): {level_counts[lvl]} nodes')

## Cell 4: Run — Business Outlook Query
**What this demonstrates**: A thematic query that cannot be answered from any single leaf chunk. RAPTOR's summary nodes — built during tree construction — provide the synthesised cross-section context that a flat index cannot offer.
**Expected output**: Level breakdown of retrieved nodes (expect summary/root nodes to dominate for this thematic query), retrieved context with tree-level labels, and the generated answer.

In [ ]:
QUERY = 'What is the overall business outlook?'

print(f'Query: {QUERY}')
print('(Thematic query — expects summary/root nodes to dominate retrieval)')
print()

result = raptor_rag(QUERY, vectorstore)

# Show which tree levels matched this query
print('Retrieved nodes by tree level:')
for lvl in sorted(result['level_breakdown'].keys()):
    count = result['level_breakdown'][lvl]
    label = 'leaf' if lvl == 0 else ('root' if lvl == max_lvl else 'summary')
    bar = '|' * count
    print(f'  Level {lvl} ({label:8s}): {count} nodes  {bar}')
print()

# Show each retrieved node with level tag and truncated text
print('Retrieved context (truncated to 180 chars per node):')
print('-' * 65)
for doc, score in result['retrieved']:
    lvl = doc.metadata['level']
    nid = doc.metadata['node_id']
    tag = 'ROOT' if lvl == max_lvl else f'L{lvl}'
    preview = doc.page_content[:180].replace('\n', ' ')
    print(f'[{tag} | {nid} | score={score:.4f}]')
    print(f'  {preview}...' if len(doc.page_content) > 180 else f'  {preview}')
    print()
print('-' * 65)
print(f'Latency: {result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])

## Cell 5: Inspect — Tree Structure and Level Match Analysis
**What this demonstrates**: Visualising the RAPTOR tree — node counts per level, sample text from each level to illustrate the abstraction progression, and a level-match analysis comparing the thematic query against a specific query to show how retrieval granularity shifts with query type.
**Expected output**: Tree structure with node counts, sample texts showing leaf-to-root abstraction, and a level-match comparison between the thematic and a specific query.

In [ ]:
# ── Tree structure overview ────────────────────────────────────────────────────
print('RAPTOR TREE STRUCTURE')
print('=' * 65)
for lvl in sorted(level_counts.keys()):
    label = 'leaf chunks      ' if lvl == 0 else ('root             ' if lvl == max_lvl else f'cluster summaries')
    bar = chr(9608) * level_counts[lvl]  # solid block character
    print(f'  Level {lvl} ({label}): {level_counts[lvl]:3d} nodes  {bar}')
print()
print(f'  Total nodes in unified index: {len(all_nodes)}')
print(f'  Leaf nodes   : {level_counts[0]}')
print(f'  Summary nodes: {len(all_nodes) - level_counts[0]}')

# ── Sample node text from each level ──────────────────────────────────────────
print()
print('ABSTRACTION PROGRESSION (sample node per level)')
print('=' * 65)
for lvl in sorted(level_counts.keys()):
    sample = next(n for n in all_nodes if n['level'] == lvl)
    label = 'leaf' if lvl == 0 else ('root' if lvl == max_lvl else 'summary')
    print(f'\nLevel {lvl} ({label}) — {sample["node_id"]}:')
    preview = sample['text'][:280].replace('\n', ' ')
    print(f'  {preview}...' if len(sample['text']) > 280 else f'  {preview}')

# ── Level match comparison: thematic vs specific query ────────────────────────
print()
print('LEVEL MATCH COMPARISON')
print('=' * 65)

SPECIFIC_QUERY = 'What was net interest income in Q3 2024?'
specific_result = raptor_rag(SPECIFIC_QUERY, vectorstore)

for label, res, qry in [
    ('THEMATIC', result, QUERY),
    ('SPECIFIC', specific_result, SPECIFIC_QUERY),
]:
    print(f'\n[{label}] "{qry}"')
    for lvl in sorted(res['level_breakdown'].keys()):
        count = res['level_breakdown'][lvl]
        lbl = 'leaf' if lvl == 0 else ('root' if lvl == max_lvl else 'summary')
        pct = 100 * count / K_RETRIEVE
        bar = chr(9608) * count
        print(f'  Level {lvl} ({lbl:8s}): {count}/{K_RETRIEVE} ({pct:.0f}%)  {bar}')

print()
thematic_summary_frac = sum(
    v for k, v in result['level_breakdown'].items() if k > 0
) / K_RETRIEVE
specific_leaf_frac = result['level_breakdown'].get(0, 0) / K_RETRIEVE

print('Observation:')
print(f'  Thematic query: {thematic_summary_frac:.0%} of retrieved nodes are summaries/root')
print('  → RAPTOR provides synthesised context no flat index can match')
print(f'  Specific query: leaf nodes dominate — precise source detail retrieved')
print('  Both queries answered from the same unified index.')

# ── Context preview ────────────────────────────────────────────────────────────
print()
print('CONTEXT SENT TO GENERATOR (thematic query, first 500 chars):')
print('-' * 65)
print(result['context'][:500])
print('...' if len(result['context']) > 500 else '')

## Cell 6: Fintech — Fund Prospectus Thematic Search
**What this demonstrates**: RAPTOR on a fund prospectus — a document type with distinct risk factor sections (credit, interest rate, liquidity, emerging market). A thematic investor query should surface summary nodes that integrate across risk sections rather than leaf nodes with specific percentages.
**Expected output**: Tree construction for the prospectus, level breakdown confirming summary-node dominance for the thematic query, and an answer synthesising risk themes across sections.

In [ ]:
PROSPECTUS = """PINNACLE EMERGING MARKETS BOND FUND
SUMMARY PROSPECTUS — JANUARY 2024

INVESTMENT OBJECTIVE
The Fund seeks to maximise total return through current income and capital
appreciation by investing primarily in fixed income securities issued by
governments and corporations in emerging market countries. The Fund targets
investors with a medium to long-term horizon who can tolerate significant
credit and currency volatility in exchange for above-benchmark yields.

PRINCIPAL INVESTMENT STRATEGIES
The Fund invests at least 80% of net assets in debt securities denominated
in U.S. dollars or local currencies of emerging market countries. The
portfolio manager allocates across Latin America, Eastern Europe, the
Middle East, Africa, and Asia-Pacific. Duration is managed within 3 to 7
years. The Fund may use currency forwards and interest rate swaps to hedge.

CREDIT RISK
The Fund invests in securities rated below investment grade, carrying
substantially higher default risk than investment-grade bonds. Issuers in
emerging markets may face deteriorating credit conditions due to commodity
price volatility, fiscal imbalances, political transition, or currency
crises. Downgrades may cause significant price declines and loss of principal.

INTEREST RATE RISK
Rising interest rates reduce bond prices. The Fund's duration exposure makes
it sensitive to global rate movements, particularly U.S. Treasury yields
which serve as the global risk-free benchmark. A 100-basis-point rise in
U.S. rates may reduce Fund NAV by approximately 4 to 6 percent.

EMERGING MARKET RISK
Political instability, expropriation, capital controls, and currency
devaluation are materially more likely in emerging markets than in developed
economies. Sanctions on sovereigns can freeze access to securities. Liquidity
in local-currency bond markets may dry up abruptly during global risk-off events.

LIQUIDITY RISK
Emerging market bonds, particularly local-currency sovereigns and corporate
bonds rated below B, may trade infrequently with wide bid-ask spreads. In
stressed market conditions the Fund may be unable to sell positions at prices
reflecting fundamental value, potentially forcing realised losses.

FEES AND EXPENSES
Management fee: 0.65% per annum. Distribution and service fees: 0.25% per
annum. Other expenses: 0.12% per annum. Total annual fund operating expenses:
1.02% of average net assets.

DISTRIBUTION POLICY
The Fund distributes net investment income monthly. Capital gains, if any,
are distributed annually. Distributions may include a return of capital,
which reduces the tax cost basis of shares held by investors."""

print('FINTECH: FUND PROSPECTUS THEMATIC SEARCH')
print()
print('Use case: asset manager screening a bond fund for suitability')
print('  Query type: thematic — expects to surface risk-section summaries')
print('  Flat retrieval would return individual risk-factor sentences;')
print('  RAPTOR summary nodes pre-integrate the risk themes.')
print()
print('Building RAPTOR tree for prospectus...')
prospectus_chunks = chunk_document(PROSPECTUS)
print(f'Leaf chunks: {len(prospectus_chunks)}')
print()

prospectus_nodes = build_raptor_tree(prospectus_chunks)
prospectus_store = build_unified_index(prospectus_nodes, collection_name='raptor_prospectus')

p_level_counts: dict[int, int] = defaultdict(int)
for node in prospectus_nodes:
    p_level_counts[node['level']] += 1
p_max_lvl = max(p_level_counts.keys())

print()
print(f'Prospectus tree: {len(prospectus_nodes)} total nodes')
for lvl in sorted(p_level_counts.keys()):
    label = 'leaf' if lvl == 0 else ('root' if lvl == p_max_lvl else 'summary')
    print(f'  Level {lvl} ({label}): {p_level_counts[lvl]} nodes')

print()
THEMATIC_QUERY = 'What are the main risk themes for an investor in this fund?'
print(f'Query: {THEMATIC_QUERY}')
print()

p_result = raptor_rag(THEMATIC_QUERY, prospectus_store)

print('Level match breakdown:')
for lvl in sorted(p_result['level_breakdown'].keys()):
    count = p_result['level_breakdown'][lvl]
    label = 'leaf' if lvl == 0 else ('root' if lvl == p_max_lvl else 'summary')
    bar = chr(9608) * count
    print(f'  Level {lvl} ({label}): {count} nodes  {bar}')

print()
top_doc, top_score = p_result['retrieved'][0]
top_lvl = top_doc.metadata['level']
top_label = 'root' if top_lvl == p_max_lvl else ('summary' if top_lvl > 0 else 'leaf')
print(f'Top retrieved node: Level {top_lvl} ({top_label}) | score={top_score:.4f}')
preview = top_doc.page_content[:300].replace('\n', ' ')
print(f'  {preview}...' if len(top_doc.page_content) > 300 else f'  {preview}')

print()
print(f'Latency: {p_result["latency_ms"]:.0f} ms')
print()
print('=' * 65)
print('PROSPECTUS THEMATIC ANSWER')
print('=' * 65)
print(p_result['answer'])
print()
print('-' * 65)
print('RAPTOR value for prospectus analysis:')
print('  Flat retrieval: individual risk-factor sentences — not synthesised')
print('  RAPTOR Level 1: integrated risk theme overview per section cluster')
print('  RAPTOR root:    fund-level investment thesis with all risks combined')
print('  One thematic query, multi-level synthesis — from the same index.')